# Administrative Geographic Hierarchy

This notebook creates a **relational table** linking Spanish municipalities to their
corresponding administrative units at multiple territorial scales:

- **Municipality** (*Municipio*)
- **Province** (*Provincia*)
- **Autonomous Community** (*Comunidad Autónoma*)
- **Agricultural Region** (*Comarca Agraria*)

This hierarchical reference table enables spatial aggregation and multi-scale analysis
across different administrative levels, and will be used to join demographic,
agricultural, and land-use datasets in subsequent analytical workflows.

The table is exported as a standalone CSV file to ensure methodological clarity,
traceability, and reproducibility.

> **Update — April 2026:** Municipal boundaries and Comarcas Agrarias now sourced
> programmatically via IGN OGC API-Features and MAPA WFS 2.0, replacing manual
> shapefile retrieval. Municipality count remains **8,132** — no boundary register
> changes recorded between 2024 and 2025. The apparent discrepancy with the INE
> Padrón histórico (t=29005, which lists 8,138 entries) reflects 6 municipalities
> suppressed before 2026 that are retained in the historical series for temporal
> continuity; they have no current geometry.

## Methodology

**Province and CCAA assignment** is extracted directly from the **nationalcode** field
returned by the IGN API, which follows the NATCODE structure:

### nationalcode structure (11 digits):
```
Position:  0-1   2-3  4-5  6-10
Example:   34     07   09   09298
Meaning: Country CCAA Prov Municipality
```

**Important:** The first 2 digits (34) represent the country code for Spain and must be
removed before extracting CCAA, Province, and Municipality codes.

Example: `34070909298`
- Country: `34` (skip this)
- CCAA: `07` (Castilla y León)
- Province: `09` (Burgos)
- Municipality: `09298`

> **Note:** Field previously named `NATCODE` in CNIG shapefiles,
> now `nationalcode` in IGN OGC API-Features response.

### Official INE Administrative Units (2025):
- **Municipalities:** 8,132 *(unchanged from 2024; no boundary register changes 2024–2025)*
- **Provinces:** 52 (50 provinces + Ceuta + Melilla)
- **Autonomous Communities:** 19 (17 CCAA + Ceuta + Melilla)

### Data Cleaning:
The IGN API returns non-municipality territories that must be excluded:
- Gibraltar (not Spanish territory)
- Plazas de soberanía (sovereignty territories with no civilian population)
- Shared land entities — mancomunidades, parzonerías, facerías (codes 53xxx–54xxx)

**Agricultural Region** assignment is performed via spatial join with MAPA Comarcas Agrarias layer.
Municipalities whose centroid falls outside comarca boundaries are assigned by maximum
area intersection. Ceuta and Melilla have no comarca by design.

**Data Sources:**
- Municipal boundaries: IGN OGC API-Features — https://api-features.ign.es/
  *(replaces manual download from Centro de Descargas CNIG)*
- Agricultural regions: MAPA WFS 2.0 — https://wmts.mapama.gob.es/sig/wfs_comun/Comarcas_Agrarias/wfs
  *(replaces local shapefile download)*

In [ ]:
"""
Notebook: 00_geographic_administrative_hierarchy.ipynb
Purpose: Create relational table linking municipalities to administrative units
Input:   IGN OGC API-Features (municipal boundaries)
         MAPA WFS 2.0 (comarcas agrarias)
Output:  mun_geographic_administrative_hierarchy.csv
         mun_geographic_administrative_hierarchy.gpkg
Author:  Juan Zotes
Last updated: 2026-04
"""

## 1. Load required libraries and configure environment

In [ ]:
# Standard library
from pathlib import Path
import re
import time
import warnings

# Third-party libraries
import pandas as pd
import geopandas as gpd
import requests

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

In [ ]:
# Base data directory
DATA_DIR = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data"
)

# Spatial derived data directory (outputs: GeoPackage, processed layers)
SPATIAL_DIR_DER = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\spatial\derived"
)

# Ensure directories exist
DATA_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_DIR_DER.mkdir(parents=True, exist_ok=True)

print(f"Data directory      : {DATA_DIR}")
print(f"Spatial derived dir : {SPATIAL_DIR_DER}")

In [ ]:
# IGN OGC API-Features — municipal boundaries
# Licence: CC BY 4.0 ign.es — https://api-features.ign.es
IGN_API_BASE = "https://api-features.ign.es"

# MAPA WFS 2.0 — Comarcas Agrarias
# Source: Ministerio de Agricultura, Pesca y Alimentación
MAPA_WFS_URL   = "https://wmts.mapama.gob.es/sig/wfs_comun/Comarcas_Agrarias/wfs"
MAPA_WFS_LAYER = "wfs_comun:Comarcas_Agrarias"

## 2a. Download municipal boundaries — IGN OGC API-Features

Municipal boundaries are downloaded programmatically from the IGN OGC API-Features service,
replacing the previous manual shapefile retrieval from the CNIG download centre.

**API endpoint:** https://api-features.ign.es/

| Layer | Collection |
|-------|-----------|
| All administrative units (municipalities, provinces, CCAA) | `administrativeunit` |

The API's `nationallevel` parameter uses full INSPIRE URIs internally and is not
reliably accepted as a query filter. All administrative units are therefore downloaded
in a single request (`country=ES`) and filtered client-side to
`nationallevelname == 'Municipio'` after download.

In [ ]:
def download_ign_municipalities(
    country: str = "ES",
    page_size: int = 500,
    max_retries: int = 3,
    retry_delay: float = 5.0,
) -> gpd.GeoDataFrame:
    """
    Download all administrative units from the IGN OGC API-Features service
    and filter to municipalities (nationallevelname == 'Municipio').

    Filtering is done client-side after download because the API's
    nationallevel parameter expects a full INSPIRE URI which is not
    reliably accepted as a query filter.

    Conforms to OGC API - Common Part 1: Core (OGC 19-072).

    Parameters
    ----------
    country : str
        ISO country filter — 'ES' for Spain.
    page_size : int
        Features per request. Keep <= 500 for IGN API stability.
    max_retries : int
        Retry attempts on transient HTTP 5xx errors.
    retry_delay : float
        Seconds between retries.

    Returns
    -------
    gpd.GeoDataFrame
        Municipalities only, EPSG:4258 (ETRS89).
        Raw download includes all AU levels; filtering applied here.
    """
    base_url = f"{IGN_API_BASE}/collections/administrativeunit/items"
    params = {
        "f":       "json",
        "limit":   page_size,
        "offset":  0,
        "country": country,
    }

    all_features = []
    total_fetched = 0
    url = base_url
    page = 1

    print("=== IGN OGC API-Features — administrativeunit ===")
    print(f"  Endpoint  : {base_url}")
    print(f"  Filter    : country={country} (all AU levels)")
    print(f"  Note      : municipality filter applied client-side")
    print(f"  Page size : {page_size}")
    print()

    while True:
        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(url, params=params, timeout=120)
                if response.status_code == 400:
                    raise ValueError("Bad request (HTTP 400). Check parameters.")
                if response.status_code == 404:
                    raise ValueError("Collection not found (HTTP 404).")
                if response.status_code >= 500:
                    raise requests.HTTPError(
                        f"Server error (HTTP {response.status_code})"
                    )
                response.raise_for_status()
                break
            except requests.HTTPError as e:
                if attempt < max_retries:
                    print(f"  ⚠ Attempt {attempt} failed: {e}. "
                          f"Retrying in {retry_delay}s...")
                    time.sleep(retry_delay)
                else:
                    raise

        data = response.json()
        features = data.get("features", [])

        if not features:
            print(f"\n  No features on page {page} — download complete.")
            break

        all_features.extend(features)
        total_fetched += len(features)
        print(f"  Page {page:>3} | {len(features):>4} features | "
              f"total: {total_fetched}", end="\r")

        # OGC API-Features §7: follow 'next' link for pagination
        next_url = next(
            (lnk["href"] for lnk in data.get("links", [])
             if lnk.get("rel") == "next"),
            None
        )
        if next_url:
            url = next_url
            params = {}
            page += 1
        else:
            print(f"\n  ✓ Raw download complete: {total_fetched} features "
                  f"in {page} page(s)")
            break

    if not all_features:
        raise RuntimeError("No features downloaded.")

    # Build GeoDataFrame
    gdf_all = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")
    gdf_all = gdf_all.to_crs("EPSG:4258")

    print(f"\n  AU levels in download:")
    for level, count in (
        gdf_all["nationallevelname"]
        .value_counts()
        .items()
    ):
        print(f"    {level:<25} : {count}")

    # Filter to municipalities only
    gdf_mun = gdf_all[
        gdf_all["nationallevelname"] == "Municipio"
    ].copy().reset_index(drop=True)

    print(f"\n  After filter (Municipio): {len(gdf_mun)} features")

    return gdf_mun


# Execute
gdf_mun_raw = download_ign_municipalities()

print(f"\n  Raw municipality count : {len(gdf_mun_raw)}")
print(f"  Columns                : {list(gdf_mun_raw.columns)}")
print(f"  CRS                    : {gdf_mun_raw.crs}")

## 2b. Download agricultural regions — MAPA WFS 2.0

Comarcas Agrarias are downloaded programmatically from the MAPA WFS 2.0 service,
replacing the previous local shapefile.

**WFS endpoint:** https://wmts.mapama.gob.es/sig/wfs_comun/Comarcas_Agrarias/wfs

| Layer | Type |
|-------|------|
| `wfs_comun:Comarcas_Agrarias` | WFS 2.0 (GeoServer) |

Paging is handled via `STARTINDEX`. Download stops when the number of returned
features is less than the requested page size.

In [ ]:
def download_mapa_comarcas(
    page_size: int = 500,
    max_retries: int = 3,
    retry_delay: float = 5.0,
) -> gpd.GeoDataFrame:
    """
    Download Comarcas Agrarias from MAPA WFS 2.0 service.

    Source: Ministerio de Agricultura, Pesca y Alimentación (MAPA)
    Service: GeoServer WFS 2.0 — https://wmts.mapama.gob.es/sig/wfs_comun/Comarcas_Agrarias/wfs
    Licence: NONE (public sector open data)

    Parameters
    ----------
    page_size : int
        Features per request (WFS 2.0 paging via STARTINDEX).
    max_retries : int
        Retry attempts on transient errors.
    retry_delay : float
        Seconds between retries.

    Returns
    -------
    gpd.GeoDataFrame
        Comarcas Agrarias in EPSG:4258 (ETRS89).
        Columns include: co_comarca, ds_comarca, co_provinc, ds_provinc, co_ccaa, ds_ccaa.
    """
    params_base = {
        "SERVICE":      "WFS",
        "VERSION":      "2.0.0",
        "REQUEST":      "GetFeature",
        "TYPENAMES":    MAPA_WFS_LAYER,
        "OUTPUTFORMAT": "application/json",
        "COUNT":        page_size,
        "STARTINDEX":   0,
    }

    all_features = []
    total_fetched = 0
    page = 1

    print("=== MAPA WFS 2.0 — Comarcas Agrarias ===")
    print(f"  Endpoint : {MAPA_WFS_URL}")
    print(f"  Layer    : {MAPA_WFS_LAYER}")
    print(f"  Page size: {page_size}")
    print()

    while True:
        params = {**params_base, "STARTINDEX": (page - 1) * page_size}

        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(
                    MAPA_WFS_URL, params=params, timeout=120
                )
                if response.status_code >= 500:
                    raise requests.HTTPError(
                        f"Server error (HTTP {response.status_code})"
                    )
                response.raise_for_status()
                break
            except requests.HTTPError as e:
                if attempt < max_retries:
                    print(f"  ⚠ Attempt {attempt} failed: {e}. "
                          f"Retrying in {retry_delay}s...")
                    time.sleep(retry_delay)
                else:
                    raise

        data = response.json()
        features = data.get("features", [])

        if not features:
            print(f"\n  No features on page {page} — download complete.")
            break

        all_features.extend(features)
        total_fetched += len(features)
        print(f"  Page {page:>3} | {len(features):>4} features | "
              f"total: {total_fetched}", end="\r")

        # WFS 2.0 paging: stop when returned < requested
        if len(features) < page_size:
            print(f"\n  ✓ Download complete: {total_fetched} features "
                  f"in {page} page(s)")
            break

        page += 1

    if not all_features:
        raise RuntimeError("No comarcas downloaded. Check WFS availability.")

    gdf = gpd.GeoDataFrame.from_features(all_features)

    # Detect CRS from response
    crs_raw = data.get("crs", {}).get("properties", {}).get("name", "EPSG:4326")
    try:
        gdf = gdf.set_crs(crs_raw, allow_override=True)
    except Exception:
        gdf = gdf.set_crs("EPSG:4326")

    gdf = gdf.to_crs("EPSG:4258")

    # Standardise column names for downstream joins
    gdf = gdf.rename(columns={
        'ds_comarca': 'Comarca_Name',
        'co_comarca': 'Comarca_Code',
    })

    print(f"\n  Columns : {list(gdf.columns)}")
    print(f"  CRS     : {gdf.crs}")

    return gdf


# Execute
print("=== Downloading Comarcas Agrarias from MAPA WFS 2.0 ===\n")
gdf_comarca = download_mapa_comarcas()
print(f"\n  ✓ Comarcas loaded: {len(gdf_comarca)}")

## 3. Extract administrative codes from nationalcode

**CRITICAL:** The `nationalcode` field contains a country code prefix (34 = Spain)
that must be removed before extracting administrative codes.

### Extraction logic:
```
Position:  0-1   2-3  4-5  6-10
Example:   34     07   09   09298
Meaning: Country CCAA Prov Municipality
```

1. Remove `ES` prefix if present
2. Skip first 2 digits (country code: 34)
3. Extract:
   - `CCAA_Code`: positions [2:4]
   - `Prov_Code`: positions [4:6]
   - `Mun_Code`: last 5 digits [-5:]

In [ ]:
# Common CRS for all layers (ETRS89 - EPSG:4258)
# This geographic CRS covers all of Spain (Peninsula, Baleares, Canarias)
TARGET_CRS = "EPSG:4258"

print(f"Target CRS: {TARGET_CRS} (ETRS89)")

In [ ]:
# API returns Peninsula, Baleares and Canarias in a single dataset — no merge required
gdf_mun = gdf_mun_raw.copy()

print(f"   Total municipalities (raw): {len(gdf_mun)}")
print(f"\nSample nationalcode structure:")
print(gdf_mun[['nationalcode', 'nameunit']].head())

In [ ]:
# VALIDATION: Check raw municipality count against INE official figure
# Raw count exceeds 8,132 — API also returns some non-municipality territories
# (shared land entities 53xxx, sovereignty territories 54xxx)
# These are filtered in Section 5 via Mun_Code < '53000'
INE_MUNICIPALITIES = 8132

try:
    assert len(gdf_mun) == INE_MUNICIPALITIES, (
        f"Expected {INE_MUNICIPALITIES} municipalities, got {len(gdf_mun)}"
    )
    print(f"✓ Municipality count matches INE figure: {INE_MUNICIPALITIES}")
except AssertionError as e:
    print(f"⚠ EXPECTED: {e}")
    print(f"   → Raw API data includes non-municipality territories")
    print(f"   → These will be excluded in Section 5 via Mun_Code < '53000'")

In [ ]:
# Confirm CRS
print(f"Target CRS: {TARGET_CRS} (ETRS89)")
print(f"Input CRS : {gdf_mun.crs}")
assert gdf_mun.crs.to_epsg() == 4258, "CRS mismatch — check download function"
print("✓ CRS confirmed\n")

# Clean nationalcode: remove 'ES' prefix if present
gdf_mun['NATCODE_str'] = (
    gdf_mun['nationalcode']
    .astype(str)
    .str.replace('ES', '', regex=False)
)

# Extract codes: skip first 2 digits (country code 34)
gdf_mun['CCAA_Code'] = gdf_mun['NATCODE_str'].str[2:4]   # Digits 3-4
gdf_mun['Prov_Code'] = gdf_mun['NATCODE_str'].str[4:6]   # Digits 5-6
gdf_mun['Mun_Code']  = gdf_mun['NATCODE_str'].str[-5:]   # Last 5 digits
gdf_mun['Mun_Name']  = gdf_mun['nameunit']

print("Codes extracted from nationalcode:")
print(gdf_mun[['nationalcode', 'NATCODE_str', 'CCAA_Code',
               'Prov_Code', 'Mun_Code', 'Mun_Name']].head(10))

## 4. Build province and CCAA lookup tables

Province and CCAA names are derived directly from the municipality dataset
using the codes extracted from `nationalcode`.
No separate province or CCAA shapefiles are required.

**Expected counts after filtering:**
- Provinces: **52** (50 provinces + Ceuta + Melilla)
- Autonomous Communities: **19** (17 CCAA + Ceuta + Melilla)

In [ ]:
# Build province lookup from municipality data
# Use only municipalities with valid codes (Mun_Code < '53000')
gdf_mun_valid = gdf_mun[gdf_mun['Mun_Code'] < '53000'].copy()

# Province lookup: Prov_Code + CCAA_Code → Prov_Name
# Use nameunit from province-level nationalcode patterns
# Since we only have municipality data, we build the lookup from
# the nationalcode structure: Prov_Name is not directly available
# — it will be loaded from the IGN administrativeunit collection
# filtered to nationallevelname == 'Provincia'

print("Building province lookup from IGN API...")

# Re-use the full AU download and filter to provinces
url_prov = f"{IGN_API_BASE}/collections/administrativeunit/items"
params_prov = {
    "f": "json",
    "limit": 100,
    "country": "ES",
}

all_au_features = []
url = url_prov
params = params_prov

while True:
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    features = data.get("features", [])
    if not features:
        break
    all_au_features.extend(features)
    next_url = next(
        (lnk["href"] for lnk in data.get("links", []) if lnk.get("rel") == "next"),
        None
    )
    if next_url:
        url = next_url
        params = {}
    else:
        break

gdf_all_au = gpd.GeoDataFrame.from_features(all_au_features, crs="EPSG:4326")

# Filter provinces
gdf_prov_raw = gdf_all_au[
    gdf_all_au['nationallevelname'] == 'Provincia'
].copy().reset_index(drop=True)

# Filter CCAA
gdf_ccaa_raw = gdf_all_au[
    gdf_all_au['nationallevelname'] == 'Comunidad autónoma'
].copy().reset_index(drop=True)

print(f"  Provinces downloaded : {len(gdf_prov_raw)}")
print(f"  CCAA downloaded      : {len(gdf_ccaa_raw)}")

In [ ]:
# Extract province codes from nationalcode
gdf_prov_raw['NATCODE_str'] = (
    gdf_prov_raw['nationalcode'].astype(str).str.replace('ES', '', regex=False)
)
gdf_prov_raw['CCAA_Code'] = gdf_prov_raw['NATCODE_str'].str[2:4]
gdf_prov_raw['Prov_Code'] = gdf_prov_raw['NATCODE_str'].str[4:6]
gdf_prov_raw['Prov_Name'] = gdf_prov_raw['nameunit']

# Province lookup table
prov_lookup = (
    gdf_prov_raw[['CCAA_Code', 'Prov_Code', 'Prov_Name']]
    .drop_duplicates(subset=['CCAA_Code', 'Prov_Code'])
    .copy()
)

print(f"Province lookup table: {len(prov_lookup)} unique provinces")
print("\nSample:")
print(prov_lookup.sort_values('Prov_Code').head(10).to_string(index=False))

In [ ]:
# Extract CCAA codes from nationalcode
gdf_ccaa_raw['NATCODE_str'] = (
    gdf_ccaa_raw['nationalcode'].astype(str).str.replace('ES', '', regex=False)
)
gdf_ccaa_raw['CCAA_Code'] = gdf_ccaa_raw['NATCODE_str'].str[2:4]
gdf_ccaa_raw['CCAA_Name'] = gdf_ccaa_raw['nameunit']

# CCAA lookup table
ccaa_lookup = (
    gdf_ccaa_raw[['CCAA_Code', 'CCAA_Name']]
    .drop_duplicates(subset=['CCAA_Code'], keep='first')
    .copy()
)

print(f"CCAA lookup table: {len(ccaa_lookup)} unique autonomous communities")
print("\nComplete list:")
print(ccaa_lookup.sort_values('CCAA_Code').to_string(index=False))

## 5. Filter non-municipality territories

The IGN API includes territories that are **NOT official municipalities**
according to INE (Instituto Nacional de Estadística):

- **Shared land entities** (codes 53xxx): mancomunidades, parzonerías, facerías
- **Plazas de soberanía** (codes 54001–54005): sovereignty territories with no civilian population
- **Gibraltar** (54006): not Spanish territory
- **Isla de los Faisanes** (54007): Hispano-French condominium

All these have `Mun_Code >= '53000'`.

### Expected counts after filtering:
- Municipalities: **8,132**
- Provinces: **52** (50 provinces + Ceuta + Melilla)
- Autonomous Communities: **19** (17 CCAA + Ceuta + Melilla)

In [ ]:
# ==============================================================================
# EXTRACT NON-MUNICIPALITY TERRITORIES (for reference before filtering)
# ==============================================================================

gdf_mun['Mun_Code'] = gdf_mun['Mun_Code'].astype(str).str.zfill(5)
gdf_mun['Mun_Code_int'] = gdf_mun['Mun_Code'].astype(int)

non_municipalities = gdf_mun[gdf_mun['Mun_Code_int'] >= 53000].copy()

shared_land = non_municipalities[
    (non_municipalities['Mun_Code_int'] >= 53000) &
    (non_municipalities['Mun_Code_int'] < 54000)
].copy()

special_territories = non_municipalities[
    non_municipalities['Mun_Code_int'] >= 54000
].copy()

mun_dict = (
    gdf_mun[gdf_mun['Mun_Code_int'] < 53000]
    .set_index('Mun_Code_int')['Mun_Name']
    .to_dict()
)

print("\n" + "="*70)
print("NON-MUNICIPAL ENTITIES (codes >= 53000)")
print("="*70)
print(f"\nTotal non-municipal entities: {len(non_municipalities)}")
print(f"  Shared land (53xxx)      : {len(shared_land)}")
print(f"  Special territories (54xxx): {len(special_territories)}")

In [ ]:
# ==============================================================================
# FILTER: KEEP ONLY OFFICIAL MUNICIPALITIES (Mun_Code < '53000')
# ==============================================================================

gdf_hierarchy = gdf_mun[
    gdf_mun['Mun_Code'] < '53000'
].copy().reset_index(drop=True)

print(f"After filtering: {len(gdf_hierarchy)} official municipalities")

In [ ]:
# ==============================================================================
# VALIDATION AGAINST OFFICIAL INE COUNTS (2025)
# ==============================================================================

INE_MUNICIPALITIES = 8132
INE_PROVINCES = 52
INE_CCAA = 19

count_mun  = gdf_hierarchy['Mun_Code'].nunique()
count_prov = gdf_hierarchy['Prov_Code'].nunique()
count_ccaa = gdf_hierarchy['CCAA_Code'].nunique()

print("\n" + "="*70)
print("VALIDATION AGAINST OFFICIAL INE COUNTS (2025)")
print("="*70)

checks_passed = True

for label, count, expected in [
    ("Municipalities", count_mun,  INE_MUNICIPALITIES),
    ("Provinces",      count_prov, INE_PROVINCES),
    ("CCAA",           count_ccaa, INE_CCAA),
]:
    if count == expected:
        print(f"✓ {label}: {count} (matches INE)")
    else:
        print(f"✗ {label}: {count} (expected {expected})")
        checks_passed = False

if checks_passed:
    print("\n✓ All counts match official INE figures")
else:
    print("\n⚠ Some counts differ from INE — review cleaning steps")

assert count_mun  == INE_MUNICIPALITIES, f"Municipality count mismatch: {count_mun}"
assert count_prov == INE_PROVINCES,      f"Province count mismatch: {count_prov}"
assert count_ccaa == INE_CCAA,           f"CCAA count mismatch: {count_ccaa}"

print("="*70)
print("✓ DATASET PASSES ALL INE CONSISTENCY CHECKS")
print("="*70)

## 6. Build administrative hierarchy table

Using the lookup tables, assign official province and CCAA names to each municipality
based on codes extracted from `nationalcode`.

In [ ]:
# ==============================================================================
# BUILD ADMINISTRATIVE HIERARCHY TABLE
# ==============================================================================

# CRITICAL: Start from gdf_hierarchy (filtered, 8132 municipalities)
# NOT from gdf_mun (unfiltered, includes 53xxx/54xxx)

print("\n" + "="*70)
print("BUILDING ADMINISTRATIVE HIERARCHY TABLE")
print("="*70)

hierarchy_df = gdf_hierarchy[['Mun_Code', 'Mun_Name', 'CCAA_Code', 'Prov_Code']].copy()

print(f"\nBase table: {len(hierarchy_df)} municipalities")

# Standardise code formats
hierarchy_df['Mun_Code']  = hierarchy_df['Mun_Code'].astype(str).str.zfill(5)
hierarchy_df['Prov_Code'] = hierarchy_df['Prov_Code'].astype(str).str.zfill(2)
hierarchy_df['CCAA_Code'] = hierarchy_df['CCAA_Code'].astype(str).str.zfill(2)

prov_lookup['Prov_Code'] = prov_lookup['Prov_Code'].astype(str).str.zfill(2)
prov_lookup['CCAA_Code'] = prov_lookup['CCAA_Code'].astype(str).str.zfill(2)
ccaa_lookup['CCAA_Code'] = ccaa_lookup['CCAA_Code'].astype(str).str.zfill(2)

print("✓ Code formats standardised (Mun: 5 digits, Prov: 2 digits, CCAA: 2 digits)")

In [ ]:
# ==============================================================================
# MERGE WITH PROVINCE NAMES
# ==============================================================================

count_before = len(hierarchy_df)

prov_lookup_clean = prov_lookup[['CCAA_Code', 'Prov_Code', 'Prov_Name']].drop_duplicates(
    subset=['CCAA_Code', 'Prov_Code'],
    keep='first'
)

hierarchy_df = hierarchy_df.merge(
    prov_lookup_clean,
    on=['CCAA_Code', 'Prov_Code'],
    how='left',
    validate='m:1'
)

count_after = len(hierarchy_df)

print(f"After province merge: {count_after} records")
print(f"Municipalities without province name: {hierarchy_df['Prov_Name'].isna().sum()}")
assert count_before == count_after, f"MERGE ERROR: Created duplicates ({count_before} -> {count_after})"
print(f"✓ No duplicates created (still {count_after} records)")

In [ ]:
# ==============================================================================
# MERGE WITH CCAA NAMES
# ==============================================================================

count_before = len(hierarchy_df)

ccaa_lookup_clean = ccaa_lookup[['CCAA_Code', 'CCAA_Name']].drop_duplicates(
    subset=['CCAA_Code'],
    keep='first'
)

hierarchy_df = hierarchy_df.merge(
    ccaa_lookup_clean,
    on='CCAA_Code',
    how='left',
    validate='m:1'
)

count_after = len(hierarchy_df)

print(f"After CCAA merge: {count_after} records")
print(f"Municipalities without CCAA name: {hierarchy_df['CCAA_Name'].isna().sum()}")
assert count_before == count_after, f"MERGE ERROR: Created duplicates ({count_before} -> {count_after})"
print(f"✓ No duplicates created (still {count_after} records)")

In [ ]:
# ==============================================================================
# VERIFY NO DUPLICATES IN HIERARCHY TABLE
# ==============================================================================

print("\n" + "="*70)
print("DUPLICATE CHECK")
print("="*70)

duplicates = hierarchy_df[hierarchy_df.duplicated(subset=['Mun_Code'], keep=False)]

if len(duplicates) > 0:
    print(f"\n⚠ CRITICAL ERROR: {len(duplicates)} duplicate Mun_Code entries found!")
    print(duplicates[['Mun_Code', 'Mun_Name', 'Prov_Name']].head(20))
    raise ValueError(f"Duplicate municipalities detected: {len(duplicates)} records")
else:
    print(f"\n✓ No duplicate municipalities")
    print(f"  Total unique Mun_Code: {hierarchy_df['Mun_Code'].nunique()}")
    print(f"  Total records: {len(hierarchy_df)}")

print("\n=== Sample hierarchical data ===")
print(hierarchy_df.head(10))

## 7. Add agricultural regions (comarcas agrarias) via spatial join

Agricultural regions are assigned via spatial join using municipality centroids.
Municipalities whose centroid falls outside comarca boundaries are assigned
by maximum area intersection.

**Note:** Ceuta and Melilla have no comarca by design.

In [ ]:
# ==============================================================================
# ADD AGRICULTURAL REGIONS (COMARCAS AGRARIAS) VIA SPATIAL JOIN
# ==============================================================================

print("\n" + "="*70)
print("ADDING AGRICULTURAL REGIONS (COMARCAS AGRARIAS)")
print("="*70)

# Prepare comarca data for spatial join
gdf_comarca_simple = gdf_comarca[['geometry', 'Comarca_Name', 'Comarca_Code']].copy()

# Compute municipality centroids from gdf_hierarchy (filtered, 8132 municipalities)
print("\nComputing municipality centroids...")
gdf_hierarchy['centroid'] = gdf_hierarchy.geometry.centroid
gdf_hierarchy_centroids = gdf_hierarchy.set_geometry('centroid')

# Spatial join
print("Performing spatial join: Municipalities → Comarcas...")
mun_comarca = gpd.sjoin(
    gdf_hierarchy_centroids[['Mun_Code', 'centroid']],
    gdf_comarca_simple,
    how='left',
    predicate='within'
)

keep_cols = ['Mun_Code', 'Comarca_Name', 'Comarca_Code']
mun_comarca = mun_comarca[keep_cols].drop_duplicates(subset=['Mun_Code'])

# Merge with hierarchy table
count_before = len(hierarchy_df)
hierarchy_df = hierarchy_df.merge(
    mun_comarca,
    on='Mun_Code',
    how='left'
)
count_after = len(hierarchy_df)

assert count_before == count_after, f"Comarca merge created duplicates: {count_before} -> {count_after}"

print(f"\n✓ Comarca data added via spatial join")
print(f"  Municipalities with comarca    : {hierarchy_df['Comarca_Name'].notna().sum()}")
print(f"  Municipalities without comarca : {hierarchy_df['Comarca_Name'].isna().sum()}")

## 8. Manual comarca assignment for edge cases

Some municipalities fail the spatial join due to:
- Centroid falling outside comarca boundaries (irregular shapes)
- Small coastal municipalities
- Autonomous cities (Ceuta, Melilla) — these have no comarca by design

Edge cases are assigned by maximum area intersection.

In [ ]:
# Analyse municipalities without comarca assignment
missing_comarca = hierarchy_df[hierarchy_df['Comarca_Name'].isna()].copy()

if len(missing_comarca) > 0:
    print("\n" + "="*70)
    print("MUNICIPALITIES WITHOUT COMARCA ASSIGNMENT")
    print("="*70)
    print(f"\nTotal: {len(missing_comarca)} municipalities")
    print("\nList:")
    for _, row in missing_comarca.iterrows():
        print(f"  {row['Mun_Code']} - {row['Mun_Name']:40s} | {row['Prov_Name']}")
else:
    print("\n✓ All municipalities have comarca assignment")

In [ ]:
# Assign comarca to municipalities without one, using area intersection
missing_comarca = hierarchy_df[hierarchy_df['Comarca_Name'].isna()].copy()

if len(missing_comarca) > 0:
    print("\n" + "="*70)
    print("ASSIGNING COMARCAS BY AREA INTERSECTION")
    print("="*70)

    assigned_count = 0

    for idx, mun_row in missing_comarca.iterrows():
        mun_code = mun_row['Mun_Code']
        mun_geom = gdf_mun[gdf_mun['Mun_Code'] == mun_code].geometry.iloc[0]

        intersecting = gdf_comarca[gdf_comarca.intersects(mun_geom)].copy()

        if len(intersecting) > 0:
            intersecting['intersection_area'] = intersecting.geometry.apply(
                lambda x: mun_geom.intersection(x).area
            )
            max_idx = intersecting['intersection_area'].idxmax()
            best_comarca = intersecting.loc[max_idx]

            hierarchy_df.loc[
                hierarchy_df['Mun_Code'] == mun_code, 'Comarca_Code'
            ] = best_comarca['Comarca_Code']
            hierarchy_df.loc[
                hierarchy_df['Mun_Code'] == mun_code, 'Comarca_Name'
            ] = best_comarca['Comarca_Name']

            print(f"  ✓ {mun_code} - {mun_row['Mun_Name']:35s} → {best_comarca['Comarca_Name']}")
            assigned_count += 1
        else:
            print(f"  ○ {mun_code} - {mun_row['Mun_Name']:35s} → No comarca (no intersection)")

    print(f"\n--- Results ---")
    print(f"  Assigned: {assigned_count}")
    print(f"  Remaining without comarca: {hierarchy_df['Comarca_Name'].isna().sum()}")

    still_missing = hierarchy_df[hierarchy_df['Comarca_Name'].isna()]
    if len(still_missing) > 0:
        print(f"\n  Final municipalities without comarca (expected: Ceuta, Melilla):")
        for _, row in still_missing.iterrows():
            print(f"    {row['Mun_Code']} - {row['Mun_Name']} ({row['Prov_Name']})")
else:
    print("\n✓ All municipalities already have comarca assignment")

## 9. Finalise hierarchy table

In [ ]:
# ==============================================================================
# FINALISE HIERARCHY TABLE
# ==============================================================================

# Select and order columns
hierarchy_final = hierarchy_df[[
    'Mun_Code',
    'Mun_Name',
    'Comarca_Code',
    'Comarca_Name',
    'Prov_Code',
    'Prov_Name',
    'CCAA_Code',
    'CCAA_Name',
]].copy()

print("\n=== Sample of final hierarchical data ===")
print(hierarchy_final.head(10))
print(f"\n--- Mun_Code format verification ---")
print(f"   Sample codes: {hierarchy_final['Mun_Code'].head(10).tolist()}")
codes_with_leading_zero = (hierarchy_final['Mun_Code'].str.startswith('0')).sum()
print(f"   Codes starting with '0' (provinces 01-09): {codes_with_leading_zero}")

## 10. Export administrative hierarchy table

In [ ]:
# ==============================================================================
# EXPORT ADMINISTRATIVE HIERARCHY TO CSV
# ==============================================================================

output_csv = SPATIAL_DIR_DER / "mun_geographic_administrative_hierarchy.csv"

hierarchy_final.to_csv(output_csv, index=False, encoding='utf-8-sig', sep=';')

print("\n" + "="*70)
print("ADMINISTRATIVE HIERARCHY EXPORTED")
print("="*70)

print(f"\n✓ CSV exported to: {output_csv}")
print(f"  Records : {len(hierarchy_final)}")
print(f"  Columns : {len(hierarchy_final.columns)}")

print(f"\n--- Column structure ---")
for i, col in enumerate(hierarchy_final.columns, 1):
    print(f"  {i}. {col}")

print(f"\n--- Final dataset ---")
print(f"  Municipalities : {len(hierarchy_final):,}")
print(f"  Expected (INE) : 8,132")
print(f"  Match          : {'✓' if len(hierarchy_final) == 8132 else '✗'}")

In [ ]:
# ==============================================================================
# EXPORT GEOPACKAGE WITH GEOMETRIES
# ==============================================================================

print("\n" + "="*70)
print("EXPORTING GEOPACKAGE")
print("="*70)

# CRITICAL: Use gdf_hierarchy (already filtered) as geometry base
gdf_hierarchy['Mun_Code'] = gdf_hierarchy['Mun_Code'].astype(str).str.zfill(5)
hierarchy_final['Mun_Code'] = hierarchy_final['Mun_Code'].astype(str).str.zfill(5)

gdf_export = gdf_hierarchy[['Mun_Code', 'Mun_Name', 'Prov_Code', 'CCAA_Code',
                             'nationalcode', 'geometry']].copy()

# Merge with hierarchy_final to add all administrative names
cols_to_add = [col for col in hierarchy_final.columns if col not in gdf_export.columns]
if cols_to_add:
    gdf_export = gdf_export.merge(
        hierarchy_final[['Mun_Code'] + cols_to_add],
        on='Mun_Code',
        how='left'
    )

# Select export columns
export_columns = [
    'Mun_Code', 'Mun_Name',
    'Comarca_Code', 'Comarca_Name',
    'Prov_Code', 'Prov_Name',
    'CCAA_Code', 'CCAA_Name',
    'nationalcode', 'geometry'
]
available_columns = [col for col in export_columns if col in gdf_export.columns]
gdf_export = gdf_export[available_columns].copy()

# Verify
print(f"\nExport verification:")
print(f"  Records in gdf_export : {len(gdf_export)}")
print(f"  Expected (INE)        : 8,132")
print(f"  Match                 : {'✓' if len(gdf_export) == 8132 else '✗'}")

sample_codes = gdf_export['Mun_Code'].head(10).tolist()
print(f"  Sample Mun_Code values: {sample_codes}")
assert all(len(str(c)) == 5 for c in sample_codes), "ERROR: Mun_Code lost leading zeros!"

# Export
output_gpkg = SPATIAL_DIR_DER / "mun_geographic_administrative_hierarchy.gpkg"
output_gpkg.parent.mkdir(parents=True, exist_ok=True)

gdf_export.to_file(
    output_gpkg,
    driver='GPKG',
    layer='municipalities_hierarchy'
)

print(f"\n✓ GeoPackage exported to:")
print(f"  {output_gpkg}")
print(f"  Layer    : municipalities_hierarchy")
print(f"  Features : {len(gdf_export)}")
print(f"  CRS      : {gdf_export.crs.to_string()}")
print(f"  Columns  : {', '.join(gdf_export.columns.tolist())}")
print(f"\n  File size: {output_gpkg.stat().st_size / 1024 / 1024:.2f} MB")

## 11. Final verification

In [ ]:
# ==============================================================================
# FINAL VERIFICATION
# ==============================================================================

print("\n" + "="*70)
print("FINAL VERIFICATION")
print("="*70)

print(f"\n=== Summary statistics ===")
print(f"  Total municipalities          : {len(hierarchy_final)}")
print(f"  Unique Mun_Code               : {hierarchy_final['Mun_Code'].nunique()}")
print(f"  Unique provinces              : {hierarchy_final['Prov_Code'].nunique()}")
print(f"  Unique CCAA                   : {hierarchy_final['CCAA_Code'].nunique()}")

if 'Comarca_Name' in hierarchy_final.columns:
    print(f"  Unique comarcas               : {hierarchy_final['Comarca_Name'].nunique()}")
    print(f"  Municipalities with comarca   : {hierarchy_final['Comarca_Name'].notna().sum()}")
    print(f"  Municipalities without comarca: {hierarchy_final['Comarca_Name'].isna().sum()}")

    missing_comarca = hierarchy_final[hierarchy_final['Comarca_Name'].isna()]
    if len(missing_comarca) > 0:
        print(f"\n  Municipalities without comarca (expected: Ceuta, Melilla):")
        for _, row in missing_comarca.iterrows():
            print(f"    • {row['Mun_Code']} - {row['Mun_Name']} ({row['Prov_Name']})")

# Mun_Code format check
codes_with_leading_zero = (hierarchy_final['Mun_Code'].str.startswith('0')).sum()
print(f"\n=== Mun_Code format check ===")
print(f"  Total codes                   : {len(hierarchy_final)}")
print(f"  Codes starting with '0' (01-09): {codes_with_leading_zero}")
print(f"  Sample codes                  : {hierarchy_final['Mun_Code'].sample(5).tolist()}")

# Final assertions
assert len(hierarchy_final) == 8132, f"Final count mismatch: {len(hierarchy_final)} vs 8132"
assert hierarchy_final['Mun_Code'].nunique() == 8132, "Duplicate Mun_Code found!"
assert (hierarchy_final['Mun_Code'].str.len() == 5).all(), "Some Mun_Code values don't have 5 digits!"

print(f"\n✓ All checks passed. Dataset ready for use.")
print("="*70)